# Will a tolerance band leave assembly clearance?

A replacement ring is heated while the shaft stays near room temperature. Predict the assembly clearance for the nominal diameters, then check the two tolerance extremes. The inputs represent an illustrative linear expansion calculation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d=50.0                         # nominal shaft and bore scale, mm
interference_low, interference_high = 22., 48.  # diametral interference, µm
alpha_ring,alpha_shaft=12e-6,11e-6             # 1/K
rise_ring,rise_shaft=100.,8.                   # K
required=10.                                  # desired diametral clearance, µm
assert d>0 and 0<=interference_low<=interference_high and min(alpha_ring,alpha_shaft)>0

## Subtract the two freely expanded diameters
A positive clearance permits assembly in the ideal diameter model. Compute the threshold from the worst initial interference, including thermal expansion of the mating shaft.

In [ ]:
def thermal_gain(dt):
    return 1000*d*(alpha_ring*dt-alpha_shaft*rise_shaft)
minimum=thermal_gain(rise_ring)-interference_high
maximum=thermal_gain(rise_ring)-interference_low
threshold=(required+interference_high+1000*d*alpha_shaft*rise_shaft)/(1000*d*alpha_ring)
assert np.isclose(thermal_gain(threshold)-interference_high,required)
print(f"Clearance range {minimum:.1f} to {maximum:.1f} µm; minimum modelled ring rise for target {threshold:.1f} K")

## Inspect the entire clearance band
The first figure assumes one shaft temperature. The second varies the shaft rise independently to show how a temperature measurement affects the inference.

In [ ]:
dt=np.linspace(0,150,160);shaft_rise=np.linspace(0,30,100)
fig,ax=plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
ax[0].fill_between(dt,thermal_gain(dt)-interference_high,thermal_gain(dt)-interference_low,alpha=.25,label="Tolerance band")
ax[0].plot(dt,thermal_gain(dt)-interference_high,label="Smallest clearance")
ax[0].axhline(required,ls="--",color="#ab532c",label="Target")
ax[0].set(xlabel="Ring rise (K)",ylabel="Diametral clearance (µm)",title="Fixed shaft temperature")
thresholds=(required+interference_high+1000*d*alpha_shaft*shaft_rise)/(1000*d*alpha_ring)
ax[1].plot(shaft_rise,thresholds,label="Worst-case threshold")
ax[1].plot(rise_shaft,threshold,"o",label="Selected temperatures")
ax[1].set(xlabel="Shaft rise (K)",ylabel="Required ring rise (K)",title="Sensitivity to shaft heating")
for p in ax: p.grid(alpha=.22);p.legend(fontsize=8)
plt.show()

Change both materials to the same expansion coefficient and give them the same temperature rise. Does the first-order clearance change? Try a nominal interference while keeping the original worst-case target. Explain why checking the nominal pair alone can miss a tight assembly.